# Zero-Shot Prompting

Issue direct task instructions with no input-output examples — ideal when the task is familiar and the model can infer format from clear constraints.


## 1. Overview

This guide covers:

- Zero-shot: instructions only, zero exemplars
- When zero-shot is sufficient vs when to add examples
- Local rubric for judging zero-shot prompt quality
- Zero-shot classification and structured extraction via Chat Completions


## 2. Motivation

Zero-shot is the default: fastest to write, fewest tokens, no example curation. Modern chat models handle many tasks (summarize, classify, translate) from a well-written instruction alone.

Reach for one-shot or few-shot when format drifts, labels are domain-specific, or evaluations show systematic errors.


## 3. Concepts

### 3.1 Glossary

| Term | Meaning |
|------|--------|
| **Zero-shot** | Prompt with task description but no input-output examples |
| **Instruction following** | Model complies from natural-language rules |
| **Label space** | Allowed categories or fields you define in the prompt |
| **Rubric** | Criteria to score output quality without golden examples |

### 3.2 How it works

The model relies on weights trained on broad internet text plus your explicit instructions. Clear label definitions and output format reduce ambiguity without showing examples.

### 3.3 When to use zero-shot

**Use when:** task is common, labels are standard, format is simple (JSON enum, yes/no, short summary).

**Avoid when:** niche taxonomy, strict schema, or repeated format errors — add examples (notebooks 06–07).

**Trade-offs:** lowest token cost and maintenance; less control than few-shot for edge cases.


## 4. Architecture

```mermaid
flowchart LR
    instr[Instructions + label defs] --> zs[Zero-shot prompt]
    input[New input text] --> zs
    zs --> llm[LLM]
    llm --> out[Label or extraction]
```

```text
[System] policy + output rules
[User]   task + definitions + <<<input>>>
         (no prior input/output pairs)
```


## 5. Local Python Examples


In [1]:
# Zero-shot rubric checker — no API required
from __future__ import annotations

from dataclasses import dataclass


@dataclass
class ZeroShotSpec:
    task: str
    labels: list[str]
    output_format: str
    constraints: str

    def render(self, input_text: str) -> str:
        label_line = ", ".join(self.labels)
        return (
            f"Task: {self.task}\n"
            f"Allowed labels: {label_line}\n"
            f"Constraints: {self.constraints}\n"
            f"Output format: {self.output_format}\n"
            f"Input:\n<<<\n{input_text.strip()}\n>>>"
        )

    def rubric_pass(self) -> list[str]:
        issues: list[str] = []
        if not self.task.strip():
            issues.append("missing task")
        if len(self.labels) < 2:
            issues.append("need at least two labels for classification")
        if "only" not in self.output_format.lower():
            issues.append("output format should say 'only' to reduce preamble")
        return issues


spec = ZeroShotSpec(
    task="Classify customer feedback sentiment.",
    labels=["positive", "neutral", "negative"],
    output_format="Reply with the label only (one word, lowercase).",
    constraints="If mixed sentiment, choose neutral.",
)

sample = "Delivery was fast but the packaging was damaged."
print(spec.render(sample))
print("rubric issues:", spec.rubric_pass() or "none")


Task: Classify customer feedback sentiment.
Allowed labels: positive, neutral, negative
Constraints: If mixed sentiment, choose neutral.
Output format: Reply with the label only (one word, lowercase).
Input:
<<<
Delivery was fast but the packaging was damaged.
>>>
rubric issues: none


## 6. OpenAI SDK Examples

```python
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(root / ".env")
client = OpenAI()
```


In [2]:
# Zero-shot classification and extraction
from __future__ import annotations

import json
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from openai import APIConnectionError, AuthenticationError, OpenAI, RateLimitError

def find_project_root(start: Path | None = None) -> Path:
    """Walk upward until requirements.txt is found (project root)."""
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find requirements.txt. Open the notebook from this repository "
        "or set the working directory to the project root."
    )

root = find_project_root()
load_dotenv(root / ".env")
client = OpenAI()
MODEL = "gpt-4o-mini"


def api_key_ready() -> bool:
    key = os.getenv("OPENAI_API_KEY", "")
    if not key.strip() or "your_openai_api_key" in key.lower():
        print("OPENAI_API_KEY not configured or still placeholder.")
        return False
    return True


def chat(system: str, user: str, *, temperature: float = 0.0) -> str | None:
    if not api_key_ready():
        return None
    try:
        response = client.chat.completions.create(
            model=MODEL,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        )
        return (response.choices[0].message.content or "").strip()
    except (AuthenticationError, RateLimitError, APIConnectionError) as exc:
        print(f"Request failed: {type(exc).__name__}: {exc}")
        return None


# --- Zero-shot classification ---
classify_user = spec.render(
    "Love the new dashboard — loads in under a second now!"
)
label = chat(
    "You classify text using only the allowed labels.",
    classify_user,
)
print("Classification:", label)

# --- Zero-shot extraction ---
extract_system = (
    "Extract structured fields from support tickets. "
    "Return valid JSON only with keys: product, issue, urgency (low|medium|high)."
)
extract_user = '''Ticket:
<<<
User cannot reset password; error 500 since 09:00 UTC. Billing portal affected.
>>>'''

raw = chat(extract_system, extract_user, temperature=0.1)
if raw:
    print("\nExtraction raw:", raw)
    try:
        # Strip optional markdown fences if model adds them
        cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip(), flags=re.I)
        data = json.loads(cleaned)
        print("Parsed JSON:", data)
    except json.JSONDecodeError as exc:
        print(f"JSON parse failed: {exc}")


Classification: positive



Extraction raw: {
  "product": "Billing portal",
  "issue": "User cannot reset password; error 500",
  "urgency": "high"
}
Parsed JSON: {'product': 'Billing portal', 'issue': 'User cannot reset password; error 500', 'urgency': 'high'}


## 7. Implementation notes

1. **`ZeroShotSpec.render`** — Keeps label definitions adjacent to the input delimiter.
2. **Temperature 0** — Prefer for classification and extraction; reduces label drift.
3. **"Label only"** — Cuts post-processing; still validate against allowed set in code.
4. **JSON extraction** — Ask for JSON only; parse defensively and handle fence wrappers.
5. **`spec` reuse** — Define once; call `render` per row in batch pipelines.


## 8. Best practices

- Name **allowed labels** explicitly; define tie-break rules ("if unsure, neutral").
- Put **definitions** for ambiguous classes in the prompt.
- Delimit **untrusted input** with `<<< >>>` or XML tags.
- Validate outputs in code (enum check, JSON schema) — do not trust free text.
- Log mispredictions; if patterns repeat, add one-shot/few-shot examples.
- Keep zero-shot prompts short; every sentence should change model behavior.


## 9. Common failure modes

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| Label plus explanation | Weak output format | "Reply with the label only" |
| Invented category | Label not in allowed set | List allowed labels; validate in code |
| JSON with prose wrapper | Format ambiguity | "Valid JSON only, no markdown" |
| Inconsistent urgency | Subjective scale | Define each level with examples in text |
| Works in demo, fails in prod | Different input distribution | Evaluate on real samples; add shots |
| High token use | Long policy in every call | Move stable rules to `system` message |


## 10. Validation checklist

1. Run rubric checker; confirm zero issues for a complete `ZeroShotSpec`.
2. Run classification on a positive sample; output is one allowed label.
3. Run extraction; JSON parses and keys match schema.
4. Break API key temporarily; confirm graceful message, no traceback.
5. Compare temperature 0 vs 0.7 on classification — confirm 0 is more stable.


## 11. Summary

- Zero-shot = clear instructions, **no examples**.
- Best for standard tasks with explicit label spaces and formats.
- Always validate outputs programmatically; upgrade to one-shot/few-shot when evals fail.

**Next:** `One_Shot_Prompting.ipynb` — steer format with a single exemplar.
